# ADAS Lane2OpenDRIVE Colab

This notebook is a thin runner. It verifies the official LATR checkpoint and runtime, then runs the repository in fail-closed mode when the legacy LATR stack is unavailable. It never substitutes dummy detections or fabricated metric geometry.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
REPO_URL = 'https://github.com/vamos-sujal/lane-opendrive.git'
REPO_DIR = Path('/content/lane-opendrive')
if REPO_DIR.exists():
    !git -C /content/lane-opendrive pull --ff-only
else:
    !git clone --depth 1 $REPO_URL /content/lane-opendrive
%cd /content/lane-opendrive

In [ ]:
import sys, torch
print(sys.version)
!nvidia-smi
assert torch.cuda.is_available(), 'A CUDA GPU is required; select a T4 runtime.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name, 'CUDA:', torch.version.cuda, 'PyTorch:', torch.__version__)
assert 'T4' in gpu_name, f'Expected NVIDIA T4, got {gpu_name}'

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt addict==2.4.0 pathspec==0.12.1
!rm -rf /content/Ultra-Fast-Lane-Detection
!git clone --depth 1 https://github.com/cfzd/Ultra-Fast-Lane-Detection.git /content/Ultra-Fast-Lane-Detection
!python scripts/verify_environment.py
import torch
assert torch.cuda.is_available(), 'CUDA is required for the T4 lane detector.'
print('UFLD source ready:', '/content/Ultra-Fast-Lane-Detection')

In [ ]:
import yaml
from pathlib import Path
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_config = yaml.safe_load(Path('configs/model.yaml').read_text())
checkpoint_name = Path(model_config['checkpoint']['path']).name
CHECKPOINT_PATH = MODEL_DIR / checkpoint_name
if not CHECKPOINT_PATH.exists():
    !python scripts/download_weights.py --config configs/model.yaml --output-dir "$MODEL_DIR"
else:
    print('Reusing checkpoint from Drive:', CHECKPOINT_PATH)
!python scripts/smoke_test.py --device cuda
print('Loaded detector: official UFLD CULane pretrained checkpoint')

In [ ]:
from pathlib import Path
from google.colab import files
INPUT_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/input')
RUNS_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/runs')
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = str(INPUT_DIR)
print('Upload option: uncomment the next lines to upload an MP4.')
# uploaded = files.upload()
# for name in uploaded:
#     Path(name).replace(INPUT_DIR / name)
print('Drive option: set VIDEO_PATH to a Drive MP4 path, or use:', INPUT_DIR)

In [ ]:
import datetime
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
RUN_DIR = RUNS_DIR / run_id
!python scripts/run_video.py --input "$VIDEO_PATH" --output "$RUN_DIR" --config configs/camera.yaml --detector ufld_culane --checkpoint "$CHECKPOINT_PATH"

In [ ]:
import json
from IPython.display import display, Image, Video

summary_path = RUN_DIR / 'run_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('=== VIDEO REPORT ===')
    print(json.dumps(summary, indent=2))
    print('\n=== IMPORTANT INTERPRETATION ===')
    print('Lane count and IDs are detector/tracker outputs.')
    print('Distance and speed are reported as unavailable unless calibrated scale/telemetry exists.')

for name in ('input_metadata.json', 'metric_report.json', 'tracking_results.json', 'lane_graph.json', 'geometry.json', 'validation_report.json'):
    path = RUN_DIR / name
    if path.exists():
        print(f'\n{name}')
        print(json.dumps(json.loads(path.read_text()), indent=2))

visual_dir = RUN_DIR / 'visualizations'
for name in ('original_video.mp4', 'lane_overlay.mp4', 'topology_overlay.mp4', 'top_down.mp4'):
    path = visual_dir / name
    if path.exists() and path.stat().st_size > 0:
        print(f'\n{name}')
        display(Video(filename=str(path), embed=False))

for path in sorted(visual_dir.glob('*_frame_*.jpg')):
    display(Image(filename=str(path)))

xodr = RUN_DIR / 'output.xodr'
print('\nOpenDRIVE:', xodr if xodr.exists() else 'not produced: metric validation failed closed')

In [ ]:
from IPython.display import display, Image
for image in sorted((RUN_DIR / 'visualizations').glob('*.png')):
    print(image.name)
    display(Image(filename=str(image)))